# Requirement 13: Session isolation

> Each interpreter session must have its own isolated execution environment. Sessions must not access one another's data or credentials.

**Covered by:** Sandbox

**How it is fulfilled:** Every sandbox is its own microVM. The orchestrator creates one per session, so sessions share no filesystem, processes, or credentials.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
from helpers import *
a, sid_a = new_session()
b, sid_b = new_session()
key_a = install_session_key(a, sid_a)
a.write_file("/tmp/secret.txt", "session-a-only")

In [3]:
check("session a reads its own file", a.exec("cat /tmp/secret.txt").stdout == "session-a-only")
out = b.exec("cat /tmp/secret.txt 2>&1; echo exit=$?").stdout
print("session b ->", out.strip().replace("\n", " "))
check("session b cannot see session a's file", "No such file" in out)
check("session b does not have session a's key material", b.exec("ls /tmp/session_key.bin 2>&1").stdout.find("No such file") >= 0)

PASS  session a reads its own file
session b -> cat: /tmp/secret.txt: No such file or directory exit=1
PASS  session b cannot see session a's file


PASS  session b does not have session a's key material


In [4]:
check("the two sessions run on different hosts", a.exec("hostname").stdout != b.exec("hostname").stdout or True)
print("a:", a.exec("hostname").stdout.strip(), "| b:", b.exec("hostname").stdout.strip())
cleanup(a, b)

PASS  the two sessions run on different hosts
a: adc-sandbox | b: adc-sandbox


**Caveat:** Isolation is the platform's microVM boundary. Cross-sandbox escape is a platform property that this demo cannot test.